<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D01_NovaBank.png"/>

# NovaBank data generation 🏦

You work as a junior data analyst at **NovaBank**, a mobile-first European bank. The risk team and the product team want to test new dashboards, but the law does not let them use real customer data. They ask you to build a synthetic dataset that looks and feels like real banking data.

You will use these files in the next exercise. So save them carefully.

## What you will produce

* `customers.csv`: 500 customer profiles
* `transactions.csv`: 5,000 transactions linked to those customers
* `loans.csv`: 800 loan applications linked to those customers

## Setup

Import the two libraries you need. Set the seed to `42`. Run this cell once before all the others.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

## Task 1: Build the customers table

You will create a DataFrame called `customers_df` with **500 rows**. Each row is one customer.

Build each column step by step in its own variable. Then put them all together in a dictionary and pass that dictionary to `pd.DataFrame`.

| Column | What it holds | Helpful functions |
| --- | --- | --- |
| `customer_id` | A whole number from 1 to 500. | `np.arange()` |
| `age` | A whole number between 18 and 80. | `np.random.randint()` |
| `gender` | One of `Male`, `Female`, `Other` with weights 48 / 48 / 4 percent. | `np.random.choice()` |
| `account_type` | One of `basic`, `premium`, `business` with weights 60 / 30 / 10 percent. | `np.random.choice()` |
| `country` | One of `France`, `Germany`, `Italy`, `Spain`, `Netherlands` with equal weights. | `np.random.choice()` |
| `is_active` | `True` 85 percent of the time, `False` 15 percent of the time. | `np.random.choice()` |
| `signup_date` | A random date between 1 January 2022 and 30 September 2025. | `pd.date_range()`, `np.random.choice()` |

In [3]:
# Number of customers
num_customers = 500

# Customer IDs from 1 to 500
customer_id = np.arange(1, num_customers + 1)

# Random ages between 18 and 80
age = np.random.randint(18, 81, size=num_customers)

# Gender with three possible values and given weights
gender_values = ['Male', 'Female', 'Other']
gender_weights = [0.48, 0.48, 0.04]
gender = np.random.choice(gender_values, size=num_customers, p=gender_weights)

# Account type with three possible values and given weights
account_values = ['basic', 'premium', 'business']
account_weights = [0.60, 0.30, 0.10]
account_type = np.random.choice(account_values, size=num_customers, p=account_weights)

# Country picked from a list with equal weights
country_values = ['France', 'Germany', 'Italy', 'Spain', 'Netherlands']
country = np.random.choice(country_values, size=num_customers)

# Active flag: 85% True, 15% False
active_values = [True, False]
active_weights = [0.85, 0.15]
is_active = np.random.choice(active_values, size=num_customers, p=active_weights)

# Signup date between 1 January 2022 and 30 September 2025
date_range = pd.date_range(start='2022-01-01', end='2025-09-30')
signup_date = np.random.choice(date_range, size=num_customers)

# Put all columns together in a dictionary
customers_data = {
    'customer_id': customer_id,
    'age': age,
    'gender': gender,
    'account_type': account_type,
    'country': country,
    'is_active': is_active,
    'signup_date': signup_date
}

# Build the DataFrame
customers_df = pd.DataFrame(customers_data)

# Check the result
print('Shape:', customers_df.shape)
print(customers_df.head())


Shape: (500, 7)
   customer_id  age  gender account_type      country  is_active signup_date
0            1   56  Female        basic       France       True  2025-05-26
1            2   69  Female      premium        Spain       True  2022-07-09
2            3   46    Male        basic  Netherlands       True  2024-09-07
3            4   32    Male     business  Netherlands       True  2023-06-30
4            5   60  Female      premium      Germany       True  2022-02-10


## Task 2: Build the transactions table

You will create a DataFrame called `transactions_df` with **5,000 rows**. Each row is one transaction made by one of the 500 customers.

Build the table the same way as Task 1: one column per variable, then a dictionary, then `pd.DataFrame`.

| Column | What it holds | Helpful functions |
| --- | --- | --- |
| `transaction_id` | A whole number from 1 to 5000. | `np.arange()` |
| `customer_id` | A whole number between 1 and 500. | `np.random.randint()` |
| `amount` | A decimal number between 1 and 2500, rounded to 2 decimals. | `np.random.uniform()`, `np.round()` |
| `transaction_type` | One of `payment`, `withdrawal`, `deposit`, `transfer` with weights 60 / 15 / 15 / 10. | `np.random.choice()` |
| `is_fraud` | `True` 2 percent of the time. | `np.random.choice()` |

In [4]:
# Number of transactions
num_transactions = 5000

# Transaction IDs from 1 to 5000
transaction_id = np.arange(1, num_transactions + 1)

# Each transaction belongs to a real customer (id between 1 and 500)
customer_id = np.random.randint(1, 501, size=num_transactions)

# Transaction amount: decimal between 1 and 2500, rounded to 2 decimals
raw_amounts = np.random.uniform(1, 2500, size=num_transactions)
amount = np.round(raw_amounts, 2)

# Transaction type with four possible values and given weights
type_values = ['payment', 'withdrawal', 'deposit', 'transfer']
type_weights = [0.60, 0.15, 0.15, 0.10]
transaction_type = np.random.choice(type_values, size=num_transactions, p=type_weights)

# Fraud flag: 2% True, 98% False
fraud_values = [True, False]
fraud_weights = [0.02, 0.98]
is_fraud = np.random.choice(fraud_values, size=num_transactions, p=fraud_weights)


| Column | What it holds | Helpful functions |
| --- | --- | --- |
| `timestamp` | A date that is always after the customer's signup date. | `pd.date_range()`, `np.random.choice()`|

A customer cannot make a transaction before they sign up. So for each transaction:

1. Find the customer's signup date in `customers_df` with setup a lookup dictionary
   like: `signup_lookup = customers_df.set_index('customer_id')['signup_date']`
2. Pick a random number of days between 1 and 700 with `np.random.randint()`.
3. Add that number of days to the signup date to get the transaction timestamp. Use `pd.Timedelta(days=...)` to add days to a date.
4. Do not exceed 30 September 2025. So put an upper limit on the number of days if needed.

You will use a simple `for` loop over the customer IDs. The loop fills a Python list with one timestamp per transaction. Then you turn the list into a Pandas Series at the end.

In [5]:
# Timestamp: each transaction must happen AFTER the customer's signup date
# Step 1: build a lookup from customer_id to signup_date
signup_lookup = customers_df.set_index('customer_id')['signup_date']

# Step 2: pick a random number of days (1 to 700) to add on top of each signup date
days_after_signup = np.random.randint(1, 701, size=num_transactions)

# Step 3: loop through each transaction and build the timestamp
max_date = pd.Timestamp('2025-09-30')
timestamp_list = []
for i in range(num_transactions):
    this_customer = customer_id[i]
    this_signup = signup_lookup[this_customer]
    this_days = days_after_signup[i]
    this_timestamp = this_signup + pd.Timedelta(days=int(this_days))
    if this_timestamp > max_date:
        this_timestamp = max_date
    timestamp_list.append(this_timestamp)

# Convert the Python list to a Pandas Series
timestamp = pd.Series(timestamp_list)

Then put all the columns together in a dictionary and create the DataFrame `transactions_df`.

In [6]:
# Put all columns in a dictionary
transactions_data = {
    'transaction_id': transaction_id,
    'customer_id': customer_id,
    'amount': amount,
    'transaction_type': transaction_type,
    'is_fraud': is_fraud,
    'timestamp': timestamp
}

# Build the DataFrame
transactions_df = pd.DataFrame(transactions_data)

# Check the result
print('Shape:', transactions_df.shape)
print(transactions_df.head())


Shape: (5000, 6)
   transaction_id  customer_id   amount transaction_type  is_fraud  timestamp
0               1          479   249.64          payment     False 2024-12-25
1               2          358  1916.72          payment     False 2023-07-06
2               3           75  2415.72          payment     False 2025-05-12
3               4          355  2425.22          payment     False 2023-01-27
4               5           72  2162.81          payment     False 2025-06-18


## Task 3: Build the loans table

You will create a DataFrame called `loans_df` with **800 rows**. Each row is one loan application from one of the 500 customers. Some customers apply more than once. That is normal.

| Column | What it holds | Helpful functions |
| --- | --- | --- |
| `loan_id` | A whole number from 1 to 800. | `range()` |
| `customer_id` | A whole number between 1 and 500. | `np.random.randint()` |
| `loan_amount` | A whole number between 500 and 50000. | `np.random.randint()` |
| `loan_type` | One of `personal`, `auto`, `home`, `business` with weights 50 / 25 / 15 / 10. | `np.random.choice()` |
| `approved` | `True` 70 percent of the time. | `np.random.choice()` |
| `application_date` | A random date between 1 January 2023 and 30 September 2025. |

In [8]:
# Number of loans
num_loans = 800

# Loan IDs from 1 to 800
loan_id = np.arange(1, num_loans + 1)

# Each loan belongs to a real customer
customer_id = np.random.randint(1, 501, size=num_loans)

# Loan amount: whole number between 500 and 50000
loan_amount = np.random.randint(500, 50001, size=num_loans)

# Loan type with four possible values and given weights
loan_type_values = ['personal', 'auto', 'home', 'business']
loan_type_weights = [0.50, 0.25, 0.15, 0.10]
loan_type = np.random.choice(loan_type_values, size=num_loans, p=loan_type_weights)

# Approval flag: 70% True, 30% False
approved_values = [True, False]
approved_weights = [0.70, 0.30]
approved = np.random.choice(approved_values, size=num_loans, p=approved_weights)

# Application date: base date + random number of days
base_loan = pd.Timestamp('2023-01-01')
random_days = np.random.randint(0, 1003, size=num_loans)
application_date = base_loan + pd.to_timedelta(random_days, unit='D')

# Put all columns in a dictionary
loans_data = {
    'loan_id': loan_id,
    'customer_id': customer_id,
    'loan_amount': loan_amount,
    'loan_type': loan_type,
    'approved': approved,
    'application_date': application_date
}

# Build the DataFrame
loans_df = pd.DataFrame(loans_data)

# Check the result
print('Shape:', loans_df.shape)
print(loans_df.head())


Shape: (800, 6)
   loan_id  customer_id  loan_amount loan_type  approved application_date
0        1          472         8174      auto     False       2023-05-18
1        2          179        31199      auto      True       2025-05-15
2        3          450        31340      auto      True       2025-06-18
3        4          240        18892      home     False       2023-11-04
4        5           12        32460  personal      True       2023-11-26


## Task 4: Save the three tables as CSV files

Save each DataFrame next to this notebook. Pass `index=False` so the row index does not end up in the file. The next exercise will load these three files using the exact names below.

In [9]:
# Save customers
customers_df.to_csv('customers.csv', index=False)
print('customers.csv saved')

# Save transactions
transactions_df.to_csv('transactions.csv', index=False)
print('transactions.csv saved')

# Save loans
loans_df.to_csv('loans.csv', index=False)
print('loans.csv saved')

customers.csv saved
transactions.csv saved
loans.csv saved


## Task 5: Quick check

Before moving to the next exercise, run a short sanity check on each table. If a shape is wrong, go back and fix the matching task. The next exercise will not work if the data is the wrong size.

In [10]:
# Customers check
print('--- customers_df ---')
print('Shape:', customers_df.shape)
print('Types:')
print(customers_df.dtypes)
print(customers_df.head(3))
print()

# Transactions check
print('--- transactions_df ---')
print('Shape:', transactions_df.shape)
print('Types:')
print(transactions_df.dtypes)
print(transactions_df.head(3))
print()

# Loans check
print('--- loans_df ---')
print('Shape:', loans_df.shape)
print('Types:')
print(loans_df.dtypes)
print(loans_df.head(3))


--- customers_df ---
Shape: (500, 7)
Types:
customer_id              int64
age                      int32
gender                     str
account_type               str
country                    str
is_active                 bool
signup_date     datetime64[us]
dtype: object
   customer_id  age  gender account_type      country  is_active signup_date
0            1   56  Female        basic       France       True  2025-05-26
1            2   69  Female      premium        Spain       True  2022-07-09
2            3   46    Male        basic  Netherlands       True  2024-09-07

--- transactions_df ---
Shape: (5000, 6)
Types:
transaction_id               int64
customer_id                  int32
amount                     float64
transaction_type               str
is_fraud                      bool
timestamp           datetime64[us]
dtype: object
   transaction_id  customer_id   amount transaction_type  is_fraud  timestamp
0               1          479   249.64          payment     False